In [1]:
"""
This file hosts the CFS URL Scanner Functions.

These functions return the URL and filename for the latest available data on the dataservers.

(C) Eric J. Drewitz 2025-2026
"""

import requests
import sys
import os
import time
import warnings 
warnings.filterwarnings('ignore')

from wxdata.utils.coords import convert_lon
from bs4 import BeautifulSoup
from wxdata.utils.nomads_gribfilter import(
    result_string,
    key_list
)

# Exception handling for Python >= 3.13 and Python < 3.13
try:
    from datetime import datetime, timedelta, UTC
except Exception as e:
    from datetime import datetime, timedelta

# Gets current time in UTC
try:
    now = datetime.now(UTC)
except Exception as e:
    now = datetime.utcnow()

# Gets local time
local = datetime.now()

# Gets yesterday's date
yd = now - timedelta(days=1)

def forecast_file_times(cat,
                        proxies:
    
    """
    This function returns the datetimes for the following:
    
    1) Final Forecast File
    2) Model Initialization
    
    Required Arguments:
    
    1) cat (String) - The category of CFS data.
    
    Categories
    ----------
    
    1) flux
    2) pressure
    
    2) proxies (dict or None) - If the user is using proxy server(s), the user must change the following:

       proxies=None ---> proxies={
                               'http':'http://your-proxy-address:port',
                               'https':'http://your-proxy-address:port'
                               }
    
    Returns
    -------
    
    The datetimes of the final forecast file and model initialization in string format.    
    """
    
    cat = cat.lower()
    
    if cat == 'flux':
        cat = 'flxf'
    else:
        cat = 'pgbf'
    
    today_18z_scan = (f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/cfs/prod/cfs.{now.strftime('%Y%m%d')}/18/6hrly_grib_01/")
    today_12z_scan = (f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/cfs/prod/cfs.{now.strftime('%Y%m%d')}/12/6hrly_grib_01/")
    today_06z_scan = (f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/cfs/prod/cfs.{now.strftime('%Y%m%d')}/06/6hrly_grib_01/")
    today_00z_scan = (f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/cfs/prod/cfs.{now.strftime('%Y%m%d')}/00/6hrly_grib_01/")

    yesterday_18z_scan = (f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/cfs/prod/cfs.{yd.strftime('%Y%m%d')}/18/6hrly_grib_01/")
    yesterday_12z_scan = (f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/cfs/prod/cfs.{yd.strftime('%Y%m%d')}/12/6hrly_grib_01/")
    yesterday_06z_scan = (f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/cfs/prod/cfs.{yd.strftime('%Y%m%d')}/06/6hrly_grib_01/")
    yesterday_00z_scan = (f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/cfs/prod/cfs.{yd.strftime('%Y%m%d')}/00/6hrly_grib_01/")

    urls = []
    urls.append(today_18z_scan)
    urls.append(today_12z_scan)
    urls.append(today_06z_scan)
    urls.append(today_00z_scan)
    urls.append(yesterday_18z_scan)
    urls.append(yesterday_12z_scan)
    urls.append(yesterday_06z_scan)
    urls.append(yesterday_00z_scan)
    
    for url in urls:    
        if proxies==None:  
            try:
                response = requests.get(url, stream=True)
            except Exception as e:
                for i in range(0, 10, 1):
                    time.sleep(60)
                    try:
                        response = requests.get(url, stream=True)
                        break
                    except Exception as e:
                        i = i
                        
        else:
            try:
                response = requests.get(url, stream=True, proxies=proxies)
            except Exception as e:
                for i in range(0, 10, 1):
                    time.sleep(60)
                    try:
                        response = requests.get(url, stream=True, proxies=proxies)
                        break
                    except Exception as e:
                        i = i
        if response.status_code == 200:
            html_content = response.text
            break
        else:
            pass   
        
    response.close()
    soup = BeautifulSoup(html_content, 'html.parser')
    file_names = set() 
    for link in soup.find_all('a', href=True):
        href = link['href']
        if '.' in href and not href.startswith('#') and not href.startswith('mailto:'):
            filename = os.path.basename(href)
            file_names.add(filename)
                
        marker = ".idx"
        fnames = []
        for filename in sorted(list(file_names)):
            if marker not in filename:
                fnames.append(filename)
        
        files = []
        for f in fnames:
            if cat in f:
                files.append(f)
                
    file = files[-1]
    date = file[4:14]
    init = file[18:26]

    if len(files) < 800:
        if now.day == local.day:
            origional = int(f"{url[-17]}{url[-16]}")
            corrected = origional - 6
            if corrected == 0:
                corrected = '00'
            elif corrected == 6:
                corrected = '06'
            else:
                corrected = str(corrected)
            new_url = f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/cfs/prod/cfs.{now.strftime('%Y%m%d')}/{corrected}/6hrly_grib_01/"

            if proxies==None:  
                try:
                    response = requests.get(new_url, stream=True)
                except Exception as e:
                    for i in range(0, 10, 1):
                        time.sleep(60)
                        try:
                            response = requests.get(new_url, stream=True)
                            break
                        except Exception as e:
                            i = i
                        
            else:
                try:
                    response = requests.get(new_url, stream=True, proxies=proxies)
                except Exception as e:
                    for i in range(0, 10, 1):
                        time.sleep(60)
                        try:
                            response = requests.get(new_url, stream=True, proxies=proxies)
                            break
                        except Exception as e:
                            i = i
            if response.status_code == 200:
                html_content = response.text
            else:
                pass   
            
            response.close()
            soup = BeautifulSoup(html_content, 'html.parser')
            file_names = set() 
            for link in soup.find_all('a', href=True):
                href = link['href']
                if '.' in href and not href.startswith('#') and not href.startswith('mailto:'):
                    filename = os.path.basename(href)
                    file_names.add(filename)
                        
                marker = ".idx"
                fnames = []
                for filename in sorted(list(file_names)):
                    if marker not in filename:
                        fnames.append(filename)
                
                files = []
                for f in fnames:
                    if cat in f:
                        files.append(f)
                        
            file = files[-1]
            date = file[4:14]
            init = file[18:26]
        else:
            yesterday_18z_scan = (f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/cfs/prod/cfs.{yd.strftime('%Y%m%d')}/18/6hrly_grib_01/")
            yesterday_12z_scan = (f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/cfs/prod/cfs.{yd.strftime('%Y%m%d')}/12/6hrly_grib_01/")
            yesterday_06z_scan = (f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/cfs/prod/cfs.{yd.strftime('%Y%m%d')}/06/6hrly_grib_01/")
            yesterday_00z_scan = (f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/cfs/prod/cfs.{yd.strftime('%Y%m%d')}/00/6hrly_grib_01/")
        
            urls = []
            urls.append(yesterday_18z_scan)
            urls.append(yesterday_12z_scan)
            urls.append(yesterday_06z_scan)
            urls.append(yesterday_00z_scan)
            
            for url in urls:    
                if proxies==None:  
                    try:
                        response = requests.get(url, stream=True)
                    except Exception as e:
                        for i in range(0, 10, 1):
                            time.sleep(60)
                            try:
                                response = requests.get(url, stream=True)
                                break
                            except Exception as e:
                                i = i
                else:
                    try:
                        response = requests.get(url, stream=True, proxies=proxies)
                    except Exception as e:
                        for i in range(0, 10, 1):
                            time.sleep(60)
                            try:
                                response = requests.get(url, stream=True, proxies=proxies)
                                break
                            except Exception as e:
                                i = i
                if response.status_code == 200:
                    html_content = response.text
                    break
                else:
                    pass   
                
            response.close()
            soup = BeautifulSoup(html_content, 'html.parser')
            file_names = set() 
            for link in soup.find_all('a', href=True):
                href = link['href']
                if '.' in href and not href.startswith('#') and not href.startswith('mailto:'):
                    filename = os.path.basename(href)
                    file_names.add(filename)
                        
                marker = ".idx"
                fnames = []
                for filename in sorted(list(file_names)):
                    if marker not in filename:
                        fnames.append(filename)
                
                files = []
                for f in fnames:
                    if cat in f:
                        files.append(f)
                        
            file = files[-1]
            date = file[4:14]
            init = file[18:26]

            if len(files) < 800:
                origional = int(f"{url[-17]}{url[-16]}")
                corrected = origional - 6
                if corrected == 0:
                    corrected = '00'
                elif corrected == 6:
                    corrected = '06'
                else:
                    corrected = str(corrected)
                new_url = f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/cfs/prod/cfs.{yd.strftime('%Y%m%d')}/{corrected}/6hrly_grib_01/"
    
                if proxies==None:  
                    try:
                        response = requests.get(new_url, stream=True)
                    except Exception as e:
                        for i in range(0, 10, 1):
                            time.sleep(60)
                            try:
                                response = requests.get(new_url, stream=True)
                                break
                            except Exception as e:
                                i = i
                else:
                    try:
                        response = requests.get(new_url, stream=True, proxies=proxies)
                    except Exception as e:
                        for i in range(0, 10, 1):
                            time.sleep(60)
                            try:
                                response = requests.get(new_url, stream=True, proxies=proxies)
                                break
                            except Exception as e:
                                i = i
                if response.status_code == 200:
                    html_content = response.text
                else:
                    pass   
                
                response.close()
                soup = BeautifulSoup(html_content, 'html.parser')
                file_names = set() 
                for link in soup.find_all('a', href=True):
                    href = link['href']
                    if '.' in href and not href.startswith('#') and not href.startswith('mailto:'):
                        filename = os.path.basename(href)
                        file_names.add(filename)
                            
                    marker = ".idx"
                    fnames = []
                    for filename in sorted(list(file_names)):
                        if marker not in filename:
                            fnames.append(filename)
                    
                    files = []
                    for f in fnames:
                        if cat in f:
                            files.append(f)
                            
                file = files[-1]
                date = file[4:14]
                init = file[18:26]
            
    
    return date, init, files

In [2]:
date, init, files = forecast_file_times('pressure', None)

In [3]:
files

['pgbf2026041312.01.2026041312.grb2',
 'pgbf2026041318.01.2026041312.grb2',
 'pgbf2026041400.01.2026041312.grb2',
 'pgbf2026041406.01.2026041312.grb2',
 'pgbf2026041412.01.2026041312.grb2',
 'pgbf2026041418.01.2026041312.grb2',
 'pgbf2026041500.01.2026041312.grb2',
 'pgbf2026041506.01.2026041312.grb2',
 'pgbf2026041512.01.2026041312.grb2',
 'pgbf2026041518.01.2026041312.grb2',
 'pgbf2026041600.01.2026041312.grb2',
 'pgbf2026041606.01.2026041312.grb2',
 'pgbf2026041612.01.2026041312.grb2',
 'pgbf2026041618.01.2026041312.grb2',
 'pgbf2026041700.01.2026041312.grb2',
 'pgbf2026041706.01.2026041312.grb2',
 'pgbf2026041712.01.2026041312.grb2',
 'pgbf2026041718.01.2026041312.grb2',
 'pgbf2026041800.01.2026041312.grb2',
 'pgbf2026041806.01.2026041312.grb2',
 'pgbf2026041812.01.2026041312.grb2',
 'pgbf2026041818.01.2026041312.grb2',
 'pgbf2026041900.01.2026041312.grb2',
 'pgbf2026041906.01.2026041312.grb2',
 'pgbf2026041912.01.2026041312.grb2',
 'pgbf2026041918.01.2026041312.grb2',
 'pgbf202604

In [13]:
initial = files[0]

In [27]:
date = f"{initial[4:14]}"
runtime = f"{initial[18:28]}"

In [33]:
runtime

'2026041312'

In [36]:
init = f"{runtime[0:8]}"

In [37]:
init

'20260413'

In [19]:
date

'2026041312'

In [20]:
date = datetime.strptime(date, '%Y%m%d%H')

In [21]:
date

datetime.datetime(2026, 4, 13, 12, 0)

In [22]:
fcst = date + timedelta(hours=384)

In [23]:
fcst

datetime.datetime(2026, 4, 29, 12, 0)

In [31]:
final = f"pgbf{fcst.strftime('%Y%m%d%H')}.01.{runtime}.grb2"

In [32]:
final

'pgbf2026042912.01.2026041312.grb2'

In [38]:
idx = files.index(final)

In [39]:
idx

64

In [41]:
files[0:(idx + 1)]

['pgbf2026041312.01.2026041312.grb2',
 'pgbf2026041318.01.2026041312.grb2',
 'pgbf2026041400.01.2026041312.grb2',
 'pgbf2026041406.01.2026041312.grb2',
 'pgbf2026041412.01.2026041312.grb2',
 'pgbf2026041418.01.2026041312.grb2',
 'pgbf2026041500.01.2026041312.grb2',
 'pgbf2026041506.01.2026041312.grb2',
 'pgbf2026041512.01.2026041312.grb2',
 'pgbf2026041518.01.2026041312.grb2',
 'pgbf2026041600.01.2026041312.grb2',
 'pgbf2026041606.01.2026041312.grb2',
 'pgbf2026041612.01.2026041312.grb2',
 'pgbf2026041618.01.2026041312.grb2',
 'pgbf2026041700.01.2026041312.grb2',
 'pgbf2026041706.01.2026041312.grb2',
 'pgbf2026041712.01.2026041312.grb2',
 'pgbf2026041718.01.2026041312.grb2',
 'pgbf2026041800.01.2026041312.grb2',
 'pgbf2026041806.01.2026041312.grb2',
 'pgbf2026041812.01.2026041312.grb2',
 'pgbf2026041818.01.2026041312.grb2',
 'pgbf2026041900.01.2026041312.grb2',
 'pgbf2026041906.01.2026041312.grb2',
 'pgbf2026041912.01.2026041312.grb2',
 'pgbf2026041918.01.2026041312.grb2',
 'pgbf202604